# Upstage AIRE LLM Deep Learning Coding Test

## Overview

This notebook is designed to assess whether the candidate can improve a specific benchmark score by fine-tuning a base LLM and whether they have the relevant engineering skills to do so.

## Task Description
* In this deep learning coding test, you are required to improve an LLM benchmark score by using Mistral-7B as a base model. You can utilize a combination of various fine-tuning strategies, including a combination of multiple datasets, Parameter Efficient Fine-Tuning (PEFT), and finding the optimal configuration of hyperparameters. The following are examples of available improvement strategies:
  * You can change the instruction template and `Tokenizer` config, etc.
  * You can implement custom classes, e.g., `CustomDataset`, `CustomCollator` and `CustomTrainer` tailored for improved instruction tuning.
  * You can explore and select high-quality datasets, or you can synthesize datasets by leveraging large-scale models (>7B).
  * You can find the optimal configuration of hyperparameters.

* We recommend using a PEFT method like QLoRA to avoid OOM errors. 7B LLMs with QLoRA-based PEFT can be trained by using a single T4 GPU (e.g., Google Colab). Note that we generally expect to achieve higher scores when using full fine-tuning compared to using PEFT.
* The notebook and source code you submit must be reproducible. For reproducibility and correct assessment, please describe the methodology you used to improve the benchmark score in as much detail as possible.
* Regarding the coding test assessment, while the primary goal is to boost your benchmark score, your methodology's logical consistency and academic rationale will also be evaluated. For example, you may fail to improve your benchmark score. Still, you may receive high marks if you analyze the reasons for your failure from both a technical and academic perspective and produce meaningful results. For more information on the benchmark score, see the Evaluation Protocol below.
* (__Important!__) The code below is provided as an example and does not guarantee the correctness of implementation and execution. You can build upon it or use a completely different code base.




## Evaluation Protocol
* ARC (AI2 Reasoning Challenge): The AI2’s Reasoning Challenge (ARC) dataset is a multiple-choice question-answering dataset, containing questions from science exams from grade 3 to grade 9. The dataset is split in two partitions: Easy and Challenge, where the latter partition contains the more difficult questions that require reasoning. Most of the questions have 4 answer choices, with $<$1\% of all the questions having either 3 or 5 answer choices. ARC includes a supporting KB of 14.3M unstructured text passages. [[source]](https://arxiv.org/abs/1911.07176)
  * Clark, Peter, et al. "Think you have solved question answering? try arc, the ai2 reasoning challenge." arXiv preprint arXiv:1803.05457 (2018). [[pdf]](https://arxiv.org/pdf/1803.05457v1.pdf)
  * The ARC Challenge benchmark has been used as an essential benchmark for major LLMs as they are released, including OpenAI's GPT-4, Google's Gemini, and Meta's LLaMA.

    source: Jiang, Albert Q., et al. "Mixtral of experts." arXiv preprint arXiv:2401.04088 (2024). [[pdf]](https://arxiv.org/pdf/2401.04088.pdf)

  * In our preliminary experiment, Mistral-7B, a base model in this notebook, has shown the ARC challenge score of 61.43.
  * In this notebook, we will utilize EleutherAI's [lm-evaluation-harness](https://github.com/EleutherAI/lm-evaluation-harness) library to evaluate the ARC challenge score.


## Submission
* Submit a notebook file and your final model (including ckpt and tokenizer). We recommend submitting the notebook as an ipynb file or Google colab, and the final model should be uploaded and shared on the HuggingFace Hub.
* Provide a separate report/documentation in PDF format where you performed detailed analysis, etc.

## Enviroment Setting

In [1]:
# Required when training models/data that are gated on Hugging Face, and required for pushing models to Hugging Face.
!pip install huggingface_hub -q -U
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [2]:
# (Optional) Mount Google Drive. If you are not using Colab, please comment out the code below.
from google.colab import drive
drive.mount('/gdrive', force_remount=True)
drive.mount('/content/drive')

Mounted at /gdrive
Mounted at /content/drive


In [3]:
# (Optional) 구글 드라이브를 사용할 경우 아래의 코드를 통해 모델을 캐싱하여 시간을 절약하고 학습 데이터를 드라이브에 저장할 수 있습니다.
# If you're running Jupyter notebook in local, set your local caching directory in `cache_dir`.

# https://stackoverflow.com/questions/56081324/why-are-google-colab-shell-commands-not-working
import locale
def getpreferredencoding(do_setlocale = True):
    return "UTF-8"
locale.getpreferredencoding = getpreferredencoding

import os
cache_dir = "/content/drive/MyDrive/huggingface_cache"
os.makedirs(cache_dir, exist_ok=True)  # Ensure the directory exists.

In [32]:
# 구글 드라이브를 사용하지 않는 경우, 아래의 코드에서 오류가 발생하는 것을 방지하기 위해 아래의 코드를 실행하세요.
# If you are not using Google Drive, please run the code below to prevent errors in the code below.
import os

cache_dir = "/tmp/huggingface_cache"

## Package & Library Install

In [4]:
!python -m pip install --upgrade pip -q -U
!pip install -q datasets
!pip install -q -U scipy
!pip install -q -U transformers
!pip install -q -U Jinja2
!pip install -q -U trl
!pip install -q -U bitsandbytes
!pip install -q -U peft
!pip install -q -U accelerate
!pip install flash-attn -q -U

  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
ERROR: Failed to build 'flash-attn' when getting requirements to build wheel


In [5]:
model_id = "mistralai/Mistral-7B-v0.1"

local_path = model_id
local_save_path = os.path.join(cache_dir, local_path)

## Download Base Model



In [6]:
from huggingface_hub import snapshot_download
import os

def download_model_repo(repo_id, local_dir):
    # Download the whole repository to the specified local directory.
    repo_path = snapshot_download(repo_id=repo_id,
                                  cache_dir=local_dir,
                                  local_dir=local_dir,
                                  local_dir_use_symlinks=False)

    print(f"Repository is saved to: {repo_path}")

def main():
    download_model_repo(model_id, local_save_path)
    print()

if __name__ == "__main__":
    main()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

Repository is saved to: /content/drive/MyDrive/huggingface_cache/mistralai/Mistral-7B-v0.1



## Load Dataset

In [20]:
from datasets import concatenate_datasets, load_dataset

ARC_DATASET_ID = "allenai/ai2_arc"
ARC_CONFIGS = ["ARC-Easy", "ARC-Challenge"]
# ARC_CONFIGS = ["ARC-Challenge"]
ARC_SPLIT = "train"
ARC_FORMAT = "question_answer"  # choices까지 넣으려면 "question_choices_answer"
ARC_SAMPLE_SIZE = 0             # 0이면 전체 ARC train 사용 (즉, 샘플 수를 지정하지 않으면 전체 사용)
SEED = 42

TRAIN_COLUMNS = ["instruction", "input", "output", "text", "source"]

def format_arc_example(example, arc_format="question_answer"):
    choices = example["choices"]
    labels = [str(label) for label in choices["label"]]
    texts = [str(text) for text in choices["text"]]

    answer = str(example["answerKey"])
    answer_text = texts[labels.index(answer)] if answer in labels else answer
    answer_text = " " + answer_text

    if arc_format == "question_choices_answer":
        formatted_choices = ". ".join(
            f"{label}. {choice_text}" for label, choice_text in zip(labels, texts)
        )
        prompt = f"{example['question']}\nChoices: {formatted_choices}\nAnswer:"
    else:
        prompt = f"{example['question']}\nAnswer:"

    return {
        "instruction": "",
        "input": prompt,
        "output": answer_text,
        "text": f"<s>Question: {prompt}{answer_text}</s>",
        "source": "arc",
    }

def format_arc_choice_contrastive_example(example, arc_format="question_answer"):
    labels = [str(label) for label in example["choices"]["label"]]
    choice_texts = [str(text) for text in example["choices"]["text"]]
    answer_key = str(example["answerKey"])

    correct_choice_index = labels.index(answer_key)
    correct_answer = choice_texts[correct_choice_index]

    if arc_format == "question_choices_answer":
        formatted_choices = ". ".join(
            f"{label}. {choice_text}"
            for label, choice_text in zip(labels, choice_texts)
        )
        prompt = f"{example['question']}\nChoices: {formatted_choices}\nAnswer:"
    else:
        prompt = f"{example['question']}\nAnswer:"

    answer_text = " " + correct_answer
    text = f"<s>Question: {prompt}{answer_text}</s>"

    return {
        "instruction": "",
        "input": prompt,
        "output": answer_text,
        "text": text,
        "source": "arc",
        "choice_prompt": f"Question: {prompt}",
        "choice_texts": [" " + choice_text for choice_text in choice_texts],
        "correct_choice_index": correct_choice_index,
    }

def sample_dataset(dataset, sample_size, seed, label):
    dataset = dataset.shuffle(seed=seed)
    if sample_size <= 0:
        print(f"Using all {len(dataset)} {label} examples.")
        return dataset
    return dataset.select(range(min(sample_size, len(dataset))))

class ChoiceContrastiveDataCollator:
    def __call__(self, features):
        return {
            "choice_prompt": [feature["choice_prompt"] for feature in features],
            "choice_texts": [list(feature["choice_texts"]) for feature in features],
            "correct_choice_index": torch.tensor(
                [int(feature["correct_choice_index"]) for feature in features],
                dtype=torch.long,
            ),
        }

arc_datasets = []

for config in ARC_CONFIGS:
    ds = load_dataset(
        ARC_DATASET_ID,
        config,
        split=ARC_SPLIT,
        cache_dir=cache_dir,
    )

    # baseline CE loss 용 데이터 처리 -> Correct answer의 생성 확률만 높임
    # ds = ds.map(
    #     format_arc_example,
    #     fn_kwargs={"arc_format": ARC_FORMAT},
    # ).select_columns(TRAIN_COLUMNS)

    # CE + Contrastive loss 용 데이터 처리 -> Correct answer의 생성 확률을 높이면서, wrong answer 보다 더 높도록 보장
    ds = ds.map(
    format_arc_choice_contrastive_example,
    fn_kwargs={"arc_format": "question_answer"},
    )

    ds = ds.remove_columns("source").add_column("source", [config] * len(ds))
    arc_datasets.append(ds)

    print(f"Loaded {len(ds)} examples from {ARC_DATASET_ID}/{config}/{ARC_SPLIT}")

sampled_arc_train_ds = concatenate_datasets(arc_datasets)
sampled_arc_train_ds = sample_dataset(sampled_arc_train_ds, ARC_SAMPLE_SIZE, SEED, "ARC")

print(sampled_arc_train_ds)
print(sampled_arc_train_ds[0]["text"])

print(sampled_arc_train_ds[0].keys())
print(sampled_arc_train_ds[0]["choice_prompt"])
print(sampled_arc_train_ds[0]["choice_texts"])
print(sampled_arc_train_ds[0]["correct_choice_index"])


Loaded 2251 examples from allenai/ai2_arc/ARC-Easy/train
Loaded 1119 examples from allenai/ai2_arc/ARC-Challenge/train
Using all 3370 ARC examples.
Dataset({
    features: ['id', 'question', 'choices', 'answerKey', 'instruction', 'input', 'output', 'text', 'choice_prompt', 'choice_texts', 'correct_choice_index', 'source'],
    num_rows: 3370
})
<s>Question: Which celestial object listed below has the greatest density?
Answer: a neutron star</s>
dict_keys(['id', 'question', 'choices', 'answerKey', 'instruction', 'input', 'output', 'text', 'choice_prompt', 'choice_texts', 'correct_choice_index', 'source'])
Question: Which celestial object listed below has the greatest density?
Answer:
[' a planet', ' a comet', ' a nebula', ' a neutron star']
3


## Load Base Model

In [21]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# Activate 4-bit precision base model loading.
use_4bit = True

# Activate nested quantization for 4-bit base models.
use_nested_quant = False


# # Compute dtype for 4-bit base models.
# bnb_4bit_compute_dtype = "float16"

# Pick exactly one mixed-precision mode end-to-end.
# bf16 does not use GradScaler; fp16 does.
bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
fp16 = torch.cuda.is_available() and not bf16
bnb_4bit_compute_dtype = "bfloat16" if bf16 else "float16"


# Quantization type (fp4 or nf4).
bnb_4bit_quant_type = "nf4"

# Load tokenizer and model with QLoRA configuration.
# compute_dtype = getattr(torch, bnb_4bit_compute_dtype)
compute_dtype = torch.bfloat16 if bf16 else torch.float16

# Enable fp16 training.
# fp16 = True

# Enable bf16 training.
# bf16 = False

bnb_config = BitsAndBytesConfig(
    load_in_4bit=use_4bit,
    bnb_4bit_quant_type=bnb_4bit_quant_type,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=use_nested_quant,
)


if compute_dtype == torch.float16 and use_4bit:
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        print("=" * 80)
        print("Your GPU supports bfloat16, you can accelerate training with the argument --bf16")
        print("=" * 80)


model = AutoModelForCausalLM.from_pretrained(
    local_save_path,
    quantization_config=bnb_config,  # Comment out to do a full fine-tune.
    device_map='auto',
    # attn_implementation="flash_attention_2",  # Comment out if you are not using an Ampere GPU (e.g. A100, H100, A6000).
    dtype=compute_dtype,  # Set to torch.float16 if you are not using an Ampere GPU (e.g. A100, H100, A6000).
    cache_dir=cache_dir)

model.config.use_cache = False
model.config.pretraining_tp = 1

tokenizer = AutoTokenizer.from_pretrained(model_id, add_eos_token=True)
tokenizer.padding_side = "right"

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [22]:
# If the tokenizer has <unk>, set the PAD token to <unk> or set it to an EOS token.
if '<pad>' in tokenizer.get_vocab():
    print('<pad> token is in the tokenizer. Using <pad> for pad')
    # Set the pad token.
    tokenizer.pad_token = '<pad>'
elif '<unk>' in tokenizer.get_vocab():
    print('<unk> token is in the tokenizer. Using unk for pad')
    # Set the pad token.
    tokenizer.pad_token = '<unk>'
else:
    print(f'Using EOS token, {tokenizer.eos_token}, for padding')
    tokenizer.pad_token = tokenizer.eos_token

<unk> token is in the tokenizer. Using unk for pad


In [23]:
model.pad_token_id = tokenizer.pad_token_id
model.config.pad_token_id = tokenizer.pad_token_id

assert model.pad_token_id == tokenizer.pad_token_id, "The model's pad token ID does not match the tokenizer's pad token ID!"

print('Tokenizer pad token ID:', tokenizer.pad_token_id)
print('Model pad token ID:', model.pad_token_id)
print('Model config pad token ID:', model.config.pad_token_id)
print('Number of tokens now in tokenizer:', tokenizer.vocab_size)

Tokenizer pad token ID: 0
Model pad token ID: 0
Model config pad token ID: 0
Number of tokens now in tokenizer: 32000


## Set Up LoRA

In [24]:
def print_trainable_parameters(model):
    """
    Outputs the number of trainable parameters in the model and the total number of parameters.
    This allows you to see the size of your model and the percentage of parameters used for training.
    """
    trainable_params = 0
    total_params = 0

    for _, param in model.named_parameters():
        total_params += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()

    trainable_percent = 100 * trainable_params / total_params

    print(f"Trainable Parameters: {trainable_params}")
    print(f"Total Parameters: {total_params}")
    print(f"Trainable %: {trainable_percent:.2f}")

In [25]:
from peft import LoraConfig, PeftModel, get_peft_model

config = LoraConfig(
    r=8,
    lora_alpha=16,
    # target_modules = ["gate_proj", "down_proj", "up_proj", "q_proj", "v_proj", "k_proj", "o_proj"]  # Targeting all modules for more intensive training.
    target_modules = ["q_proj", "v_proj"],  # There are options to deepen the fine-tuning by unfreezing more weights but with a cost in performance.
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

from peft import prepare_model_for_kbit_training

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)
ft_model = get_peft_model(
    model,
    config,
)
print_trainable_parameters(ft_model)

Trainable Parameters: 3407872
Total Parameters: 3755479040
Trainable %: 0.09


## Set Hyperparameters

In [26]:
from trl import SFTConfig, SFTTrainer

# pad_token이 없을 수 있음
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

training_arguments = SFTConfig(
    output_dir=os.path.join(cache_dir, "results"),
    num_train_epochs=2,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    optim="paged_adamw_32bit",
    save_steps=50,
    logging_steps=10,
    learning_rate=1e-4,
    weight_decay=0.001,
    bf16=bf16,
    fp16=fp16,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=0.03,
    lr_scheduler_type="constant",
    dataset_text_field="text",
    max_length=1024,
    packing=False,
)


import trl, transformers, peft, accelerate, datasets, torch
print("trl", trl.__version__)
print("transformers", transformers.__version__)
print("peft", peft.__version__)
print("accelerate", accelerate.__version__)
print("datasets", datasets.__version__)
print("torch", torch.__version__)


[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


trl 1.3.0
transformers 5.7.0
peft 0.19.1
accelerate 1.13.0
datasets 4.8.5
torch 2.10.0+cu128


In [ ]:
# Setting sft parameters.
# baseline CE loss 용 trainer
# trainer = SFTTrainer(
#     model=model,
#     train_dataset=sampled_arc_train_ds,
#     peft_config=config,
#     processing_class=tokenizer,
#     args=training_arguments,
# )


In [27]:
from transformers import Trainer

class ChoiceContrastiveTrainer(Trainer):
    def __init__(
        self,
        *args,
        choice_tokenizer,
        temperature=1.0,
        max_length=1024,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)
        self.choice_tokenizer = choice_tokenizer
        self.temperature = temperature
        self.max_length = max_length

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        prompts = inputs["choice_prompt"]
        choice_texts = inputs["choice_texts"]
        correct_choice_index = inputs["correct_choice_index"].to(model.device)

        choice_counts = [len(choices) for choices in choice_texts]
        flat_prompts = []
        flat_choices = []

        for prompt, choices in zip(prompts, choice_texts):
            flat_prompts.extend([prompt] * len(choices))
            flat_choices.extend(choices)

        flat_scores = self._compute_length_normalized_choice_scores(
            model,
            flat_prompts,
            flat_choices,
        )

        max_choices = max(choice_counts)
        choice_scores = flat_scores.new_full(
            (len(choice_counts), max_choices),
            -torch.inf,
        )

        offset = 0
        for row_idx, count in enumerate(choice_counts):
            choice_scores[row_idx, :count] = flat_scores[offset:offset + count]
            offset += count

        logits = choice_scores / self.temperature
        loss = torch.nn.functional.cross_entropy(logits, correct_choice_index)

        return (loss, {"logits": logits}) if return_outputs else loss

    def _compute_length_normalized_choice_scores(self, model, prompts, choices):
        prefixes = [f"<s>{prompt}" for prompt in prompts]
        full_texts = [
            f"{prefix}{choice}"
            for prefix, choice in zip(prefixes, choices)
        ]

        prefix_encodings = self.choice_tokenizer(
            prefixes,
            add_special_tokens=False,
            truncation=True,
            max_length=self.max_length,
        )

        prompt_lengths = torch.tensor(
            [len(input_ids) for input_ids in prefix_encodings["input_ids"]],
            device=model.device,
            dtype=torch.long,
        )

        encodings = self.choice_tokenizer(
            full_texts,
            add_special_tokens=False,
            padding=True,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        ).to(model.device)

        outputs = model(
            input_ids=encodings["input_ids"],
            attention_mask=encodings["attention_mask"],
        )

        logits = outputs.logits[:, :-1, :]
        target_ids = encodings["input_ids"][:, 1:]
        target_attention = encodings["attention_mask"][:, 1:].bool()

        positions = torch.arange(target_ids.shape[1], device=model.device).unsqueeze(0)

        # Token at target position j corresponds to original input token j + 1.
        # Answer tokens start at original token index prompt_length.
        answer_mask = positions >= (prompt_lengths.unsqueeze(1) - 1)
        answer_mask = answer_mask & target_attention

        answer_token_counts = answer_mask.sum(dim=1)
        if torch.any(answer_token_counts == 0):
            raise ValueError(
                "At least one answer was truncated away. "
                "Increase max_length or shorten the prompt."
            )

        token_logprobs = torch.log_softmax(logits, dim=-1).gather(
            dim=-1,
            index=target_ids.unsqueeze(-1),
        ).squeeze(-1)

        sum_logprobs = (token_logprobs * answer_mask).sum(dim=1)

        # Contrastive loss에 answer length normalize
        avg_logprobs = sum_logprobs / answer_token_counts

        return avg_logprobs

In [28]:
training_arguments.remove_unused_columns = False

trainer = ChoiceContrastiveTrainer(
    model=ft_model,  # or model, depending on your notebook variable name
    train_dataset=sampled_arc_train_ds,
    args=training_arguments,
    data_collator=ChoiceContrastiveDataCollator(),
    choice_tokenizer=tokenizer,
    temperature=1.0,
    max_length=1024,
    processing_class=tokenizer,  # use tokenizer=tokenizer if your transformers version is older
)

## Fine-Tuning

In [29]:
import torch
from collections import Counter

print(torch.cuda.get_device_name(0))
print(torch.cuda.get_device_capability(0))
print("bf16 supported:", torch.cuda.is_bf16_supported())

print("args fp16:", trainer.args.fp16)
print("args bf16:", trainer.args.bf16)

try:
    print("accelerate mixed_precision:", trainer.accelerator.state.mixed_precision)
    print("scaler:", trainer.accelerator.scaler)
except Exception as e:
    print(e)

# cnt = Counter()
# for name, p in trainer.model.named_parameters():
#     if p.requires_grad:
#         cnt[str(p.dtype)] += p.numel()
#         print(name, p.dtype, p.shape)
# print(cnt)


NVIDIA A100-SXM4-80GB
(8, 0)
bf16 supported: True
args fp16: False
args bf16: True
accelerate mixed_precision: bf16
scaler: None


In [30]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Step,Training Loss
10,1.015036
20,0.695216
30,0.651672
40,0.415761
50,0.539204
60,0.646473
70,0.570293
80,0.558513
90,0.813462
100,0.603628


TrainOutput(global_step=1686, training_loss=0.44126808508117005, metrics={'train_runtime': 820.6852, 'train_samples_per_second': 8.213, 'train_steps_per_second': 2.054, 'total_flos': 0.0, 'train_loss': 0.44126808508117005, 'epoch': 2.0})

## Save and Push Model to HuggingFace Hub

In [31]:
merged_model = trainer.model.merge_and_unload()

/usr/local/lib/python3.12/dist-packages/peft/tuners/lora/bnb.py:373: UserWarning: Merge lora module to 4-bit linear may get different generations due to rounding errors.
  warnings.warn(


In [32]:
new_model_name = "Mongjin/mistral-7b-qlora-arc-nce_colab"  # Please specify your own repo/model id.

In [33]:
tokenizer_to_save = AutoTokenizer.from_pretrained(
    model_id,
    add_eos_token=False,
    trust_remote_code=True,
)

tokenizer_to_save.padding_side = "right"
if "<pad>" in tokenizer_to_save.get_vocab():
    tokenizer_to_save.pad_token = "<pad>"
elif "<unk>" in tokenizer_to_save.get_vocab():
    tokenizer_to_save.pad_token = "<unk>"
else:
    tokenizer_to_save.pad_token = tokenizer_to_save.eos_token

output_merged_dir = os.path.join(cache_dir, new_model_name)
merged_model.save_pretrained(output_merged_dir, safe_serialization=True)
tokenizer_to_save.save_pretrained(output_merged_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/huggingface_cache/Mongjin/mistral-7b-qlora-arc-nce_colab/tokenizer_config.json',
 '/content/drive/MyDrive/huggingface_cache/Mongjin/mistral-7b-qlora-arc-nce_colab/tokenizer.json')

In [34]:
# Push merged model to the HF hub.
merged_model.push_to_hub(repo_id=new_model_name, token=True, max_shard_size="5GB")
tokenizer_to_save.push_to_hub(repo_id=new_model_name, token=True)

README.md:   0%|          | 0.00/31.0 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...hb30w5b/model.safetensors:   2%|2         |  104MB / 4.98GB            

CommitInfo(commit_url='https://huggingface.co/Mongjin/mistral-7b-qlora-arc-nce_colab/commit/086fcc6999690e7f7f9a5d6ab67b1ce2e1ecc4f9', commit_message='Upload tokenizer', commit_description='', oid='086fcc6999690e7f7f9a5d6ab67b1ce2e1ecc4f9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Mongjin/mistral-7b-qlora-arc-nce_colab', endpoint='https://huggingface.co', repo_type='model', repo_id='Mongjin/mistral-7b-qlora-arc-nce_colab'), pr_revision=None, pr_num=None)

In [ ]:
del model, merged_model
torch.cuda.empty_cache()

## ARC Challenge Evaluation

If you get an OOM error in the evaluation command below when using Google Colab, try restarting your session and rerunning the cells, except for training parts.

In [35]:
!git clone https://github.com/EleutherAI/lm-evaluation-harness
!cd lm-evaluation-harness && pip install -e .

fatal: destination path 'lm-evaluation-harness' already exists and is not an empty directory.
Obtaining file:///content/lm-evaluation-harness
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for lm_eval (pyproject.toml) ... done
  Created wheel for lm_eval: filename=lm_eval-0.4.12.dev0-0.editable-py3-none-any.whl size=29241 sha256=e479a37342bc3124960b4dab0e351d06a96c8fbdaeff7472529191fc98231f72
  Stored in directory: /tmp/pip-ephem-wheel-cache-3i0ugrn0/wheels/8b/3b/a3/4f9491fc7a02bc58df7511185ce9669bec3daf1f8c214b0907
Successfully built lm_eval
  Attempting uninstall: lm_eval
    Found existing installation: lm_eval 0.4.12.dev0
    Uninstalling lm_eval-0.4.12.dev0:
      Successfully uninstalled lm_eval-0.4.12.dev0


In [36]:
# Mistral-7B's ARC Challenge Score: 61.43
# eval_output_path = os.path.join(cache_dir, "results", "arc_challenge")
# os.makedirs(eval_output_path, exist_ok=True)

# # It takes about 11 minutes on a single A100 40GB GPU (about 100 minutes on a single T4 GPU)
# eval_output_path = os.path.join(eval_output_path, "result.json")
# tasks = "arc_challenge"

# eval_cmd = f"""
# lm_eval --model hf \
#     --model_args pretrained=mistralai/Mistral-7B-v0.1,trust_remote_code=True,dtype=float16 \
#     --tasks {tasks} \
#     --device cuda:0 \
#     --batch_size 8 \
#     --num_fewshot 25 \
#     --output_path {eval_output_path}
# """

"""
hf (pretrained=mistralai/Mistral-7B-v0.1,trust_remote_code=True,dtype=float16), gen_kwargs: (None), limit: None, num_fewshot: 25, batch_size: 8
|    Tasks    |Version|Filter|n-shot| Metric |Value |   |Stderr|
|-------------|------:|------|-----:|--------|-----:|---|-----:|
|arc_challenge|      1|none  |    25|acc     |0.5700|±  |0.0145|
|             |       |none  |    25|acc_norm|0.6143|±  |0.0142|
"""

'\nhf (pretrained=mistralai/Mistral-7B-v0.1,trust_remote_code=True,dtype=float16), gen_kwargs: (None), limit: None, num_fewshot: 25, batch_size: 8\n|    Tasks    |Version|Filter|n-shot| Metric |Value |   |Stderr|\n|-------------|------:|------|-----:|--------|-----:|---|-----:|\n|arc_challenge|      1|none  |    25|acc     |0.5700|±  |0.0145|\n|             |       |none  |    25|acc_norm|0.6143|±  |0.0142|\n'

In [37]:
eval_output_path = os.path.join(cache_dir, "results", "arc_challenge")
os.makedirs(eval_output_path, exist_ok=True)

# It takes about 11 minutes on a single A100 40GB GPU (about 100 minutes on a single T4 GPU).
eval_output_path = os.path.join(eval_output_path, "result-25shot.json")
tasks = "arc_challenge"

eval_cmd = f"""
lm_eval --model hf \
    --model_args pretrained={new_model_name},trust_remote_code=True,dtype=float16 \
    --tasks {tasks} \
    --device cuda:0 \
    --batch_size 8 \
    --num_fewshot 25 \
    --output_path {eval_output_path}
"""

In [38]:
# Run an evaluation command.
!{eval_cmd}

2026-04-29:05:41:58 INFO     [_cli.run:378] Selected Tasks: ['arc_challenge']
2026-04-29:05:42:00 INFO     [evaluator:213] Setting random seed to 0 | Setting numpy seed to 1234 | Setting torch manual seed to 1234 | Setting fewshot manual seed to 1234
2026-04-29:05:42:00 INFO     [evaluator:238] Initializing hf model, with arguments: {'pretrained': 'Mongjin/mistral-7b-qlora-arc-nce_colab', 'trust_remote_code': True, 'dtype': 'float16'}
2026-04-29:05:42:05 INFO     [models.huggingface:256] Using device 'cuda:0'
config.json: 1.19kB [00:00, 2.66MB/s]
tokenizer_config.json: 100% 493/493 [00:00<00:00, 1.08MB/s]
tokenizer.json: 3.51MB [00:00, 25.8MB/s]
2026-04-29:05:42:08 INFO     [models.huggingface:524] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
/usr/local/lib/python3.12/dist-packages/transformers/quantizers/auto.py:262: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're load

The final model shows the ARC Challenge score of 58.19 (the value corresponding to acc_norm). This is worse than the pre-fine-tuning score of 61.43 (in our preliminary study), indicating that a better strategy is needed to improve the score.

In [ ]:
!pip freeze > requirements.txt

In [ ]:
!cat requirements.txt

## Content License

Copyright : <font color='blue'> <b> ©2024 by Upstage Co., Ltd. All rights reserved.</font></b>

<font color='red'><b>WARNING</font> : 본 콘텐츠의 지식재산권은 업스테이지에 귀속됩니다. 본 콘텐츠를 어떠한 경로로든 외부로 유출 및 수정하는 행위를 엄격히 금합니다. </b>